# Hierarchical Bayesian shape + scale (empirical Bayes)

**Date:** 2026-04-18.
**Motivation:** Shape×Scale failed because cluster shape was used as-is for the target. Hierarchical shape partially pools target's OWN observed arrivals with cluster prior, letting the target's data override cluster when it differs materially.

**Architecture:**
```
Each movie has Gamma(α, β) arrival-time shape (days since first review).
Cluster prior:     weighted-average (α, β) across similar training movies.
Target MLE:        Gamma fit on target's observed arrivals.
Pooled posterior:  θ = w_emp × target + (1 − w_emp) × cluster
                   w_emp = n_obs / (n_obs + prior_strength)
F_target(t) = gamma.cdf with pooled params → V inferred, prediction follows.
```

Compare to: weighted-KDE (midnight+noon), Ridge, shape×scale (no pooling).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gamma as gamma_dist
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    critic_activity_counts, observed_review_stats,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SNAP_DAYS = 3

reviews_noon = reviews.copy()
day_mask = reviews_noon['timestamp_confidence'] == 'd'
reviews_noon.loc[day_mask, 'estimated_timestamp'] = (
    reviews_noon.loc[day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)
first_review_ts_noon = (reviews_noon[reviews_noon['movie_slug'].isin(close_date_map)]
                         .groupby('movie_slug')['estimated_timestamp'].min())

activity = critic_activity_counts()
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']

CACHE = CACHE_DIR / 'hier_bayes.pkl'
print('Ready.')

## Per-movie arrival times and Gamma fits

In [ ]:
# Build per-movie arrival times (days since first review, noon-shifted)
movie_arrivals = {}
for slug in close_date_map:
    if slug not in first_review_ts_noon.index:
        continue
    first_ts = first_review_ts_noon.loc[slug]
    close_ts = close_date_map[slug]
    mr = reviews_noon[(reviews_noon['movie_slug'] == slug)
                      & (reviews_noon['estimated_timestamp'] < close_ts)]
    if len(mr) == 0:
        continue
    deltas = (mr['estimated_timestamp'] - first_ts).dt.total_seconds() / 86400
    # Add tiny epsilon to handle t=0 (first review itself)
    arr = np.sort(deltas.values)
    arr = arr + 1e-6
    movie_arrivals[slug] = arr

print(f'Built arrivals for {len(movie_arrivals)} movies')


def fit_gamma_mom(arrivals):
    """Method of moments Gamma(α, β) fit. Returns (α, β) or None."""
    if len(arrivals) < 3:
        return None
    mu = np.mean(arrivals)
    var = np.var(arrivals)
    if var <= 0 or mu <= 0:
        return None
    α = mu * mu / var
    β = mu / var
    return float(α), float(β)


# Per-movie Gamma params
movie_gamma = {}
for slug, arr in movie_arrivals.items():
    params = fit_gamma_mom(arr)
    if params is not None:
        movie_gamma[slug] = params

print(f'Fit Gamma for {len(movie_gamma)} movies')
α_vals = [p[0] for p in movie_gamma.values()]
β_vals = [p[1] for p in movie_gamma.values()]
print(f'α distribution: mean={np.mean(α_vals):.2f} median={np.median(α_vals):.2f} std={np.std(α_vals):.2f}')
print(f'β distribution: mean={np.mean(β_vals):.2f} median={np.median(β_vals):.2f} std={np.std(β_vals):.2f}')

## Cluster prior from combined_score-weighted training

In [ ]:
def cluster_gamma_prior(scores):
    """Weighted average of Gamma params across training movies."""
    items = [(s, w) for s, w in scores.items() if s in movie_gamma]
    if not items:
        return None
    total_w = sum(w for _, w in items)
    if total_w <= 0:
        return None
    α_cluster = sum(movie_gamma[s][0] * w for s, w in items) / total_w
    β_cluster = sum(movie_gamma[s][1] * w for s, w in items) / total_w
    return α_cluster, β_cluster

## Hierarchical prediction

In [ ]:
def hier_predict(slug, prior_strength=40.0):
    target_close = close_date_map[slug]
    midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400
    close_midnight = target_close.floor('D')

    mr_all = reviews_noon[reviews_noon['movie_slug'] == slug]
    obs = mr_all[(mr_all['estimated_timestamp'] < snap_time) & (mr_all['estimated_timestamp'] < target_close)]
    if len(obs) < 3:
        return None
    first_ts = obs['estimated_timestamp'].min()
    first_review_dbc = (target_close - first_ts).total_seconds() / 86400
    obs_window_days = first_review_dbc - snap_dbc_eff
    if obs_window_days <= 0:
        return None
    target_gap = gap_for_slug(slug)
    if target_gap is None:
        return None

    # Target observed arrivals (days since first review)
    target_arrivals = (obs['estimated_timestamp'] - first_ts).dt.total_seconds().values / 86400 + 1e-6
    target_arrivals = np.sort(target_arrivals)
    target_mle = fit_gamma_mom(target_arrivals)

    # Cluster prior
    scores = combined_score_with_scores(
        slug, target_gap, set(obs['reviewer_name']), obs_window_days,
        k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
    )
    if len(scores) < 5:
        return None
    cluster_prior = cluster_gamma_prior(scores)
    if cluster_prior is None:
        return None

    # Pool
    n_obs = len(obs)
    if target_mle is None:
        α_post, β_post = cluster_prior
    else:
        w_emp = n_obs / (n_obs + prior_strength)
        α_post = w_emp * target_mle[0] + (1 - w_emp) * cluster_prior[0]
        β_post = w_emp * target_mle[1] + (1 - w_emp) * cluster_prior[1]

    # CDF at observation end and prediction end (both in days since first review)
    t_obs_end = obs_window_days
    t_pred_end = first_review_dbc - midnight_utc_dbc

    F_obs = gamma_dist.cdf(t_obs_end, a=α_post, scale=1.0/β_post)
    F_pred = gamma_dist.cdf(t_pred_end, a=α_post, scale=1.0/β_post)

    if F_obs <= 0:
        return None
    V = n_obs / F_obs
    predicted = V * (F_pred - F_obs)

    actual = int(((mr_all['estimated_timestamp'] >= snap_time) & (mr_all['estimated_timestamp'] < close_midnight)).sum())

    return {
        'slug': slug, 'n_obs': n_obs, 'first_review_dbc': first_review_dbc,
        'target_α_mle': target_mle[0] if target_mle else np.nan,
        'target_β_mle': target_mle[1] if target_mle else np.nan,
        'cluster_α': cluster_prior[0], 'cluster_β': cluster_prior[1],
        'α_post': α_post, 'β_post': β_post,
        'F_obs': float(F_obs), 'F_pred': float(F_pred),
        'V': float(V), 'pred': float(predicted), 'actual': actual,
        'err': float(predicted) - actual,
    }

## Prior-strength sweep

In [ ]:
if CACHE.exists():
    all_rows = pd.read_pickle(CACHE)
    print(f'Loaded {len(all_rows)} cached rows')
else:
    rows = []
    for prior_strength in [10, 20, 40, 80, 160]:
        for slug in close_date_map:
            r = hier_predict(slug, prior_strength=prior_strength)
            if r is not None:
                r['prior_strength'] = prior_strength
                rows.append(r)
    all_rows = pd.DataFrame(rows)
    all_rows.to_pickle(CACHE)
    print(f'Cached {len(all_rows)} rows')

all_rows['abs_err'] = all_rows['err'].abs()

print('Aggregate MAE by prior_strength:\n')
print(f'  {"prior":>8s}  {"cohort_MAE":>11s}  {"cohort_me":>10s}  {"h/m_MAE":>8s}  {"h/m_me":>8s}')
for ps in [10, 20, 40, 80, 160]:
    sub = all_rows[all_rows['prior_strength'] == ps]
    cohort = sub[~sub['slug'].isin(HM)]
    hm_sub = sub[sub['slug'].isin(HM)]
    print(f'  {ps:>8d}  {cohort["abs_err"].mean():>11.2f}  {cohort["err"].mean():>+10.2f}  {hm_sub["abs_err"].mean():>8.2f}  {hm_sub["err"].mean():>+8.2f}')

## Stratify by actual quartile

In [ ]:
# Stratify at prior_strength=40 (middle)
ps_focus = 40
sub = all_rows[all_rows['prior_strength'] == ps_focus]
sub['q_actual'] = pd.qcut(sub['actual'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')

print(f'Stratified MAE at prior_strength={ps_focus}:\n')
for q in ['Q1','Q2','Q3','Q4']:
    qsub = sub[sub['q_actual'] == q]
    if len(qsub):
        lo, hi = int(qsub['actual'].min()), int(qsub['actual'].max())
        print(f'  {q} (actual [{lo}, {hi}])  n={len(qsub):3d}  MAE={qsub["abs_err"].mean():6.2f}  mean_err={qsub["err"].mean():+6.2f}')

## H/m per-target comparison + baselines

In [ ]:
# Compute baselines: weighted-KDE and Ridge
def wkde_predict(slug):
    target_close = close_date_map[slug]
    midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400
    obs = reviews_noon[(reviews_noon['movie_slug']==slug) & (reviews_noon['estimated_timestamp']<snap_time)]
    if len(obs) < 3:
        return None
    target_gap = gap_for_slug(slug)
    if target_gap is None:
        return None
    state = {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': float((target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400),
    }
    if state['first_review_dbc'] < snap_dbc_eff + 1.0:
        return None
    tw = state['first_review_dbc'] - snap_dbc_eff
    scores = combined_score_with_scores(
        slug, target_gap, state['observed_critics'], tw,
        k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
    )
    if len(scores) < 5:
        return None
    try:
        profiles = build_weighted_critic_profiles(reviews_noon, close_date_map, scores, verbose=False)
        if len(profiles.df) == 0:
            return None
        model = build_weighted_kde_lambda_model(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)
        return float(predict_window_custom(
            model, dbc_from=snap_dbc_eff, dbc_to=midnight_utc_dbc,
            observed_critics=state['observed_critics'],
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        ))
    except Exception:
        return None


hier = all_rows[all_rows['prior_strength'] == ps_focus].copy()
hier['wkde_pred'] = hier['slug'].apply(wkde_predict)

# Ridge (simple, CV holding h/m separate)
FEATURES_RIDGE = ['n_obs', 'first_review_dbc']  # minimal subset for quick compute
# Actually use the full feature set — rebuild here
def features_for(slug):
    td = hier[hier['slug']==slug]
    if len(td) == 0:
        return None
    row = td.iloc[0]
    target_close = close_date_map[slug]
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    close_midnight = target_close.floor('D')
    mr = reviews_noon[reviews_noon['movie_slug']==slug]
    obs = mr[(mr['estimated_timestamp']<snap_time) & (mr['estimated_timestamp']<target_close)]
    if len(obs) < 3:
        return None
    first_ts = obs['estimated_timestamp'].min()
    obs_window = (snap_time - first_ts).total_seconds() / 86400
    last_day_start = snap_time - pd.Timedelta(days=1)
    rate_last = ((obs['estimated_timestamp']>=last_day_start) & (obs['estimated_timestamp']<snap_time)).sum()
    first_end = first_ts + pd.Timedelta(days=1)
    rate_first = ((obs['estimated_timestamp']>=first_ts) & (obs['estimated_timestamp']<first_end)).sum()
    stats = observed_review_stats(slug, first_ts, obs_window, activity)
    return {
        'slug': slug, 'observed_count': len(obs),
        'first_review_dbc': row['first_review_dbc'],
        'target_gap': gap_for_slug(slug),
        'observed_rate': len(obs)/obs_window,
        'rate_last_day': int(rate_last), 'rate_first_day': int(rate_first),
        'top_critic_frac': stats['top_critic_frac'], 'pub_diversity': stats['pub_diversity'],
        'pub_entropy': stats['pub_entropy'], 'low_activity_frac': stats['low_activity_frac'],
        'actual': row['actual'],
    }

feat_rows = [features_for(s) for s in hier['slug']]
feat_rows = [r for r in feat_rows if r is not None]
feat = pd.DataFrame(feat_rows)

FEATURES = ['observed_count','first_review_dbc','target_gap','observed_rate',
            'rate_last_day','rate_first_day','top_critic_frac','pub_diversity','pub_entropy','low_activity_frac']
cohort_feat = feat[~feat['slug'].isin(HM)]
hm_feat = feat[feat['slug'].isin(HM)]
X_c = cohort_feat[FEATURES].values
y_c = cohort_feat['actual'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
ridge_cohort = np.zeros(len(cohort_feat))
for tr, te in kf.split(X_c):
    m = Ridge(alpha=10.0); m.fit(X_c[tr], y_c[tr])
    ridge_cohort[te] = m.predict(X_c[te])
m_all = Ridge(alpha=10.0); m_all.fit(X_c, y_c)
ridge_hm = m_all.predict(hm_feat[FEATURES].values)
ridge_map = dict(zip(cohort_feat['slug'], ridge_cohort))
for s, p in zip(hm_feat['slug'], ridge_hm):
    ridge_map[s] = p
hier['ridge_pred'] = hier['slug'].map(ridge_map)

print('\nH/m per-target comparison (prior_strength=40):\n')
cols = ['slug', 'actual', 'n_obs', 'α_post', 'β_post', 'V', 'F_obs', 'F_pred', 'pred', 'wkde_pred', 'ridge_pred']
print(hier[hier['slug'].isin(HM)][cols].to_string(index=False, float_format='%.2f'))

# Full comparison summary
print('\nFull comparison (cohort / h/m):\n')
def mae(preds, actuals):
    err = preds - actuals
    return err.abs().mean(), err.mean()

for label, scope in [('Cohort (no h/m)', hier[~hier['slug'].isin(HM)]), ('H/m', hier[hier['slug'].isin(HM)])]:
    hier_mae, hier_me = mae(scope['pred'], scope['actual'])
    wk_mae, wk_me = mae(scope['wkde_pred'].dropna(), scope[scope['wkde_pred'].notna()]['actual'])
    rd_mae, rd_me = mae(scope['ridge_pred'], scope['actual'])
    print(f'  {label:16s}  hier_MAE={hier_mae:5.2f} me={hier_me:+5.2f}  |  wkde_MAE={wk_mae:5.2f} me={wk_me:+5.2f}  |  ridge_MAE={rd_mae:5.2f} me={rd_me:+5.2f}')

## Visualize target-specific vs cluster vs posterior shape for the_drama

In [ ]:
target = 'the_drama'
r = hier_predict(target, prior_strength=40.0)
if r is not None:
    α_mle, β_mle = r['target_α_mle'], r['target_β_mle']
    α_cl, β_cl = r['cluster_α'], r['cluster_β']
    α_po, β_po = r['α_post'], r['β_post']

    ts = np.linspace(0.1, 10, 300)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(ts, gamma_dist.cdf(ts, a=α_cl, scale=1/β_cl), label=f'cluster prior  α={α_cl:.2f} β={β_cl:.2f}', linestyle='--')
    ax.plot(ts, gamma_dist.cdf(ts, a=α_mle, scale=1/β_mle), label=f'target MLE  α={α_mle:.2f} β={β_mle:.2f}', linestyle=':')
    ax.plot(ts, gamma_dist.cdf(ts, a=α_po, scale=1/β_po), label=f'posterior (pooled)  α={α_po:.2f} β={β_po:.2f}', linewidth=2.5)
    target_close = close_date_map[target]
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    obs_end = r['first_review_dbc'] - (target_close - snap_time).total_seconds()/86400
    pred_end = r['first_review_dbc'] - (target_close - target_close.floor('D')).total_seconds()/86400
    ax.axvline(obs_end, color='gray', linestyle=':', alpha=0.5, label=f'obs_end={obs_end:.2f}')
    ax.axvline(pred_end, color='black', linestyle=':', alpha=0.5, label=f'pred_end={pred_end:.2f}')
    ax.set_xlabel('Days since first review')
    ax.set_ylabel('CDF')
    ax.set_title(f'{target}: cluster vs MLE vs posterior Gamma shapes')
    ax.legend()
    plot_path = ROOT.parent / 'notebooks' / 'hier_bayes_the_drama.png' if ROOT.name == 'notebooks' else ROOT / 'notebooks' / 'hier_bayes_the_drama.png'
    plt.tight_layout()
    plt.savefig(plot_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved {plot_path}')
    print(f'\nPosterior prediction: V={r["V"]:.1f} × (F_pred={r["F_pred"]:.3f} − F_obs={r["F_obs"]:.3f}) = {r["pred"]:.2f}')
    print(f'Actual: {r["actual"]}')

## Decision

- **Hier wins h/m and cohort or at least h/m without cohort regression** → partial pooling is the right architecture; worth further investment.
- **Hier wins h/m calibration only** → worth considering as part of blend.
- **Hier loses everywhere** → shape assumption (Gamma) may be mis-specified; or prior strength is wrong; or partial-pooling is not enough given cohort limitations.